In [ ]:
from typo_utils import make_typo, ngram_overlap, evaluate_ngram_overlap_mecab, compute_micro_average, ipadic_files, compute_metrics
from llama_utils import get_llama_server_models
import MeCab
import random
import pandas as pd
from tqdm.notebook import tqdm
import typo_utils
import copy

In [ ]:
from llama_utils import get_llama_server_models
model_name = get_llama_server_models("http://localhost:8080")["data"][0]["id"]
print(model_name)

In [ ]:
import json
import re
import requests

def strip_gpt_oss_channels(s: str) -> str:
    """llama-server(gpt-oss)がcontentに混ぜる <|channel|>... を除去して本文だけ返す"""
    if not s:
        return ""

    # final チャンネルがある場合は final だけ採用
    m = re.search(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|end\|>|$)", s, re.DOTALL)
    if m:
        return m.group(1).strip()

    # analysis + <|end|> + 本文 の形式なら <|end|> 以降だけ採用
    if "<|end|>" in s:
        return s.split("<|end|>")[-1].strip()

    # それ以外：タグを雑に除去
    s = re.sub(r"<\|[^>]*\|>", "", s)
    return s.strip()

def extract_last_report(text: str) -> str:
    """テンプレが2回出る等の事故に備え、最後のヘッダ以降だけ採用"""
    if not text:
        return ""
    header = "【乳癌取扱い規約・第19版.】"
    idxs = [m.start() for m in re.finditer(re.escape(header), text)]
    if not idxs:
        return text.strip()
    return text[idxs[-1]:].strip()

def fix_with_llama_server_chat(
    selected: dict,
    server_url: str = "http://127.0.0.1:8080/v1/chat/completions",
) -> str:
    system_prompt = """あなたは病理診断レポート作成者です。
【厳守】
- 出力は次のテンプレートを埋めた本文のみ（1回）とする
- 前置き、解説、注釈、英語、Markdown、箇条書き、コードブロック、思考過程を一切出力しない
- テンプレートに無い行を追加しない
- 値が不明な場合は「不明」と記載する（空欄にしない）
- テンプレートを写す行為（下書き）をしない。完成した1回分のみ出力する。
- T因子は「浸潤径」欄に記載された数値（mm）のみを使用し、その最大値で判定する。
  「浸潤径+乳管内進展巣」はT因子の判定に使用しない。
  判定は次の境界条件を厳密に守ること：
  最大浸潤径 ≤1mm → pT1mi
  1mm＜最大浸潤径 ≤5mm → pT1a
  5mm＜最大浸潤径 ≤10mm → pT1b
  10mm＜最大浸潤径 ≤20mm → pT1c
  20mm＜最大浸潤径 ≤50mm → pT2
  50mm＜最大浸潤径 → pT3
- 核グレードは「核異型スコア」と
  「核分裂像スコア（核グレード分類）」を加算した合計点のみで判定する。
  他の情報は一切考慮しない。
  合計点が2点または3点 → 1
  合計点が4点 → 2
  合計点が5点または6点 → 3
- 組織学的グレードは
  「腺管形成スコア」＋「核異型スコア」＋
  「核分裂像スコア（組織学的グレード分類）」の合計点のみで判定する。
  他の情報は一切考慮しない。
  合計点が3点, 4点, 5点 → 1
  合計点が6点, 7点 → 2
  合計点が8点, 9点 → 3 """

    template = """【乳癌取扱い規約・第19版.】
術前治療の有無: {{術前治療の有無}}.
術式: {{術式}} + {{リンパ節郭清}}.
腫瘍の部位: {{左右}}/{{乳房内局在}}.
浸潤径: {{浸潤径}}.
浸潤径+乳管内進展巣: {{浸潤径+乳管内進展巣}}.
組織型: {{組織型}}.
T因子: {{T因子}}. リンパ管侵襲: {{リンパ管侵襲}}. 静脈侵襲: {{静脈侵襲}}.
脂肪浸潤: {{脂肪浸潤}}. 皮膚浸潤: {{皮膚浸潤}}.
核グレード分類: Grade {{核グレード}} (核異型スコア: {{核異型スコア}}点, 核分裂像スコア: {{核分裂像スコア（核グレード分類）}}点).
組織学的グレード分類: Grade {{組織学的グレード}} (腺管形成スコア: {{腺管形成スコア}}点, 核異型スコア: {{核異型スコア}}点, 核分裂像スコア: {{核分裂像スコア（組織学的グレード分類）}}点)."""

    user_prompt = f"""次のJSONの値だけを使って、下のテンプレートを埋めて出力してください。

テンプレート（このままの改行・記号で出力）：
{template}

JSON:
{json.dumps(selected, ensure_ascii=False, indent=2)}
"""

    payload = {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        #"temperature": 0.1,
         "temperature": 1.0,
        "top_p": 1.0,
        "presence_penalty": 0.0,
    }

    try:
        r = requests.post(
            server_url,
            json=payload,
            proxies={"http": None, "https": None},
            timeout=600,
        )
        r.raise_for_status()
        data = r.json()

        content = (data.get("choices", [{}])[0].get("message", {}) or {}).get("content", "")
        content = content.strip()

        # gpt-oss特有のチャンネルタグ等を除去
        content = strip_gpt_oss_channels(content)
        
        return content

    except requests.exceptions.RequestException as e:
        print("❌ LLM接続エラー:", e)
        # 可能ならレスポンス本文も出す
        try:
            print("response text:", r.text)  # type: ignore
        except Exception:
            pass
        return ""

In [ ]:
import itertools

chars = 'ABCDE'
all_patterns = []
for r in range(1, len(chars) + 1):
    perms = itertools.permutations(chars, r)
    all_patterns.extend(''.join(p) for p in perms)

def make_invasive_size():
    a = round(random.uniform(0.1, 40), 1)
    b = round(random.uniform(0.1, 40), 1)
    c = round(random.uniform(0, 100), 1)
    return f"{a}×{b}×{c}mm"

def make_entire_size(size_str):
    # 例: "12.3x24.5x8.1mm" → ['12.3', '24.5', '8.1']
    try:
        base = size_str.replace('mm', '').split('×')
        a, b, c = map(float, base)
    except Exception as e:
        raise ValueError("サイズ文字列の形式が正しくありません。'AxBxCmm' 形式で指定してください。") from e

    # 各値より大きい値をランダムに生成
    a_prime = round(random.uniform(a, max(a + 10, a + 0.2)), 1)
    b_prime = round(random.uniform(b, max(b + 10, b + 0.2)), 1)
    c_prime = round(random.uniform(c, max(c + 20, c + 0.2)), 1)

    return f"{a_prime}×{b_prime}×{c_prime}mm"

import re
from typing import Optional

def breast_pt_from_size_str(size: str) -> Optional[str]:
    max_mm = _max_dimension_mm(size)
    if max_mm is None:
        return None

    if max_mm <= 1:
        return "pT1mi"
    elif max_mm <= 5:
        return "pT1a"
    elif max_mm <= 10:
        return "pT1b"
    elif max_mm <= 20:
        return "pT1c"
    elif max_mm <= 50:
        return "pT2"
    else:
        return "pT3"

def _max_dimension_mm(size: str) -> Optional[float]:
    """
    '15×12×8mm' のような文字列から数値を抜き出し、最大値(mm)を返す
    """

    if not isinstance(size, str):
        return None

    s = size.lower()

    # mm を除去（あってもなくてもOK）
    s = s.replace("mm", "")

    # ×, x, X で区切られた数値をすべて拾う
    nums = re.findall(r"\d+(?:\.\d+)?", s)
    if not nums:
        return None

    return max(float(n) for n in nums)

def calc_nuclear_grade(a, b):
    total = a + b
    if total<4:
        return 1
    elif total==4:
        return 2
    else:
        return 3

def calc_histological_grade(a, b, c):
    total = a + b + c
    if total<6:
        return 1
    elif total in [6, 7]:
        return 2
    else:
        return 3
    

data = {
    '術前治療の有無': ['無し', '有り'],
    '術式': ['Bp', 'Bp', 'Bt', 'Md', 'SSM', 'NSM'],
    'リンパ節郭清': ['SN→Ax(Ⅰ)', 'SN→Ax(Ⅱ)', 'SN→Ax(Ⅲ)'],
    '左右': ['左', '右', '左'],
    '乳房内局在': all_patterns,
    '組織型': [
                 'Invasive ductal carcinoma, tubule forming type',
                 'Invasive ductal carcinoma, tubule forming type',
                 'Invasive ductal carcinoma, solid type',
                 'Invasive ductal carcinoma, scirrhous type',
                 'Invasive ductal carcinoma, other type',
                 'Invasive ductal carcinoma, status post-chemotherapy',
                 'Invasive lobular carcinoma',
                 'Ductal carcinoma in situ(DCIS)',
                 'Lobular carcinoma in situ(LCIS)',
                 'Microinvasive carcinoma',
                 'Tubular carcinoma',
                 'Invasive cribriform carcinoma',
                 'Mucinous carcinoma',
                 'Medullary carcinoma',
                 'Apocrine carcinoma',
                 'Squamous cell carcinoma',
                 'Carcinoma with mesenchymal differentiation',
                 'Spindle cell carcinoma',
                 'Carcinoma with osseous/cartilaginous differentiation',
                 'Matrix-producing carcinoma',
                 'Invasive micropapillary carcinoma',
                 'Secretory carcinoma',
                 'Adenoid cystic carcinoma',
                 "Paget's disease"],
    'リンパ管侵襲': ['Ly0', 'Ly1', 'LyX'],
    '静脈侵襲': ['V0', 'V1', 'VX'],
    '脂肪浸潤': ['f(-)', 'f(+)'],
    '皮膚浸潤': ['s(-)', 's(+)'],
    '核異型スコア': [1, 2, 3],
    '核分裂像スコア（核グレード分類）': [1, 2, 3],
    '腺管形成スコア': [1, 2, 3],
    '核分裂像スコア（組織学的グレード分類）': [1, 2, 3],
}

In [ ]:
import re
from typing import Optional, Dict

def extract_pt_nuclear_histologic_grade(text: str) -> Dict[str, Optional[str]]:
    """
    LLM出力テキストから以下3つを抽出:
      - T因子 (例: pT1c)
      - 核グレード (例: Grade 1)
      - 組織学的グレード (例: Grade 2)

    返り値例:
      {"T因子": "pT1c", "核グレード": "1", "組織学的グレード": "2"}
    見つからない場合は None
    """
    if not isinstance(text, str):
        return {"T因子": None, "核グレード": None, "組織学的グレード": None}

    # 余分な空白をならす（全角スペース等も軽く吸収）
    s = re.sub(r"\s+", " ", text)

    # T因子: "T因子: pT1c." のような形を想定（末尾の句点はあってもなくてもOK）
    m_t = re.search(r"T因子\s*:\s*(pT1mi|pTis|pT[0-4][a-c]?)", s, re.IGNORECASE)
    t_factor = m_t.group(1) if m_t else None

    # 核グレード分類: "核グレード分類: Grade 1 (...)" を想定
    m_ng = re.search(r"核グレード分類\s*:\s*Grade\s*([123])", s, re.IGNORECASE)
    nuclear_grade = m_ng.group(1) if m_ng else None

    # 組織学的グレード分類: "組織学的グレード分類: Grade 2 (...)" を想定
    m_hg = re.search(r"組織学的グレード分類\s*:\s*Grade\s*([123])", s, re.IGNORECASE)
    histologic_grade = m_hg.group(1) if m_hg else None

    return {
        "T因子": t_factor,
        "核グレード": nuclear_grade,
        "組織学的グレード": histologic_grade,
    }

def calc_correct(pred, gold):
    try:
        NG = int(pred["核グレード"])
        HG = int(pred["組織学的グレード"])
        return pred["T因子"] == gold["T因子"], NG==gold["核グレード"], HG==gold["組織学的グレード"]
    except:
        return pred["T因子"] == gold["T因子"], pred["核グレード"]==gold["核グレード"], pred["組織学的グレード"]==gold["組織学的グレード"]

In [ ]:
def sample_selected():
    selected = {key: random.choice(values) for key, values in data.items()}
    grade1 = calc_nuclear_grade(selected['核異型スコア'], selected['核分裂像スコア（核グレード分類）'])
    grade2 = calc_histological_grade(selected['腺管形成スコア'], selected['核異型スコア'], selected['核分裂像スコア（組織学的グレード分類）'])
    invasive_size = make_invasive_size()
    entire_size = make_entire_size(invasive_size)
    selected['浸潤径'] = invasive_size
    selected['浸潤径+乳管内進展巣'] = entire_size
    selected_for_llm = copy.deepcopy(selected)
    
    selected['T因子'] = breast_pt_from_size_str(invasive_size)
    selected['核グレード'] = grade1
    selected['組織学的グレード'] = grade2
    return selected_for_llm, selected
def make_formatted_report(selected):
    breast_format = f"""【乳癌取扱い規約・第19版.】
    術前治療の有無: {selected['術前治療の有無']}.
    術式: {selected['術式']} + {selected['リンパ節郭清']}.
    腫瘍の部位: {selected['左右']}/{selected['乳房内局在']}.
    浸潤径: {selected['浸潤径']}.
    浸潤径+乳管内進展巣: {selected['浸潤径+乳管内進展巣']}.
    組織型: {selected['組織型']}.
    T因子: {selected['T因子']}. リンパ管侵襲: {selected['リンパ管侵襲']}. 静脈侵襲: {selected['静脈侵襲']}.
    脂肪浸潤: {selected['脂肪浸潤']}. 皮膚浸潤: {selected['皮膚浸潤']}.
    核グレード分類: Grade {selected['核グレード']} (核異型スコア: {selected['核異型スコア']}点, 核分裂像スコア: {selected['核分裂像スコア（核グレード分類）']}点).
    組織学的グレード分類: Grade {selected['組織学的グレード']} (腺管形成スコア: {selected['腺管形成スコア']}点, 核異型スコア: {selected['核異型スコア']}点, 核分裂像スコア: {selected['核分裂像スコア（組織学的グレード分類）']}点).
    """
    return breast_format

def overall_accuracy(results):
    """
    results: [(bool, bool, bool), ...]
    3項目すべてを1判定としてのAccuracy
    """
    if not results:
        return 0.0

    correct = sum(all(r) for r in results)
    return correct / len(results)
def per_item_accuracy(results):
    """
    各項目（T因子 / 核グレード / 組織学的グレード）のAccuracy
    """
    if not results:
        return (0.0, 0.0, 0.0)

    n = len(results)
    t_acc = sum(r[0] for r in results) / n
    n_acc = sum(r[1] for r in results) / n
    h_acc = sum(r[2] for r in results) / n

    return t_acc, n_acc, h_acc

In [ ]:
dic_dir = "./dictionaries/mecab-ipadic-2.7.0-20070610"
usr_dir = "./dictionaries/user.dic"

tagger = MeCab.Tagger(f'-r /dev/null -d "{dic_dir}" -u "{usr_dir}"')


In [ ]:
import time
results_mecab_with_counts = []
results_char_with_counts = []
index = list(range(100))
llama_times = []
ans_pT_NG_HG = []


for progress, i in tqdm(enumerate(index, start=1), total=len(index) ):
    selected_for_llm, selected = sample_selected()
    original = typo_utils.normalize_unicode_symbol_white( make_formatted_report(selected).strip() )
    t0 = time.perf_counter()
    fixed = fix_with_llama_server_chat(selected_for_llm, server_url="http://localhost:8080/v1/chat/completions").strip()
    t1 = time.perf_counter()
    llama_times.append(t1 - t0)
    fixed = typo_utils.normalize_unicode_symbol_white( fixed )

    pred = extract_pt_nuclear_histologic_grade(fixed)
    
    ans_pT_NG_HG += [calc_correct(pred, selected)]    
    # MeCabベースのn-gram評価
    metrics_mecab = evaluate_ngram_overlap_mecab(
        ref_text=original,
        pred_text=fixed,
        tagger=tagger,
        n=2,
        return_counts=True
    )
    results_mecab_with_counts.append(metrics_mecab)

    # 文字n-gram評価
    metrics_char = ngram_overlap(
        ref=original,
        pred=fixed,
        n=3,
        return_counts=True
    )
    results_char_with_counts.append(metrics_char)

    if metrics_mecab['f1']<0.8:
        print(metrics_mecab)
        print(f"[{progress}] オリジナル:")
        print(original)
        print(f"[{progress}] LLM出力:")
        print(fixed)
        print()

    # ログ表示（100件ごと）
    if progress % 10 == 0 or progress==1:
        compute_metrics(results_mecab_with_counts, results_char_with_counts)
        print(f"[{progress}] オリジナル:")
        print(original)
        print(f"[{progress}] LLM出力:")
        print(fixed)
        print()
        print(overall_accuracy(ans_pT_NG_HG))
        print(per_item_accuracy(ans_pT_NG_HG))
        
        
compute_metrics(results_mecab_with_counts, results_char_with_counts)
print(overall_accuracy(ans_pT_NG_HG))
print(per_item_accuracy(ans_pT_NG_HG))

In [ ]:
import numpy as np
print(f"llama中央値応答時間: {np.median(llama_times):.3f} 秒")
print(f"llama平均応答時間: {np.mean(llama_times):.3f} 秒")
print(f"llama最小応答時間: {np.min(llama_times):.3f} 秒")
print(f"llama最大応答時間: {np.max(llama_times):.3f} 秒")